# DiD-BCF — B1_selection_obs (linearity_degree=1)

**Workstream B1 · canonical DiD (selection on unobservables)**

continuity check: selection on observables only

Fits DiD-BCF on the `B1_selection_obs` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.0 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_selection_obs",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_selection_obs_lin_1] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_selection_obs: 100%|██████████| 100/100 [6:48:36<00:00, 245.17s/fit]

[B1_selection_obs_lin_1] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_selection_obs_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_selection_obs,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.283333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
1,canonical,B1_selection_obs,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.402667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
2,canonical,B1_selection_obs,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.332667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
3,canonical,B1_selection_obs,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.393333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677
4,canonical,B1_selection_obs,1,200,0,ES,k=0,NaN,NaN,0.0,...,0.283333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.924677


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_selection_obs,1,200,ATT,ATT,power,-0.631152,-3.933742,0.27,...,0.964273,1.375880,0.735536,3.933742,1.0,0.0,0.891748,3.935424,0.379556,5.266589
1,canonical,B1_selection_obs,1,200,ES,k=0,power,-0.622404,-3.927066,0.35,...,1.214377,1.233427,0.743600,3.927066,1.0,0.0,0.918048,3.928128,0.443888,29.294897
2,canonical,B1_selection_obs,1,200,ES,k=1,power,-0.601174,-3.932039,0.39,...,1.186314,1.422741,0.696953,3.932039,1.0,0.0,0.855777,3.933917,0.485428,4.686699
3,canonical,B1_selection_obs,1,200,ES,k=2,power,-0.662865,-3.940222,0.34,...,1.192689,1.556540,0.759644,3.940222,1.0,0.0,0.914571,3.942386,0.464467,4.237540
4,canonical,B1_selection_obs,1,200,ES,k=3,power,-0.638164,-3.935642,0.36,...,1.260237,1.674856,0.780911,3.935642,1.0,0.0,0.943542,3.938390,0.454061,3.879902
5,canonical,B1_selection_obs,1,200,GATT,g=4_t=4,power,-0.622404,-3.927066,0.35,...,1.214377,1.233427,0.743600,3.927066,1.0,0.0,0.918048,3.928128,0.443888,29.294897
6,canonical,B1_selection_obs,1,200,GATT,g=4_t=5,power,-0.601174,-3.932039,0.39,...,1.186314,1.422741,0.696953,3.932039,1.0,0.0,0.855777,3.933917,0.485428,4.686699
7,canonical,B1_selection_obs,1,200,GATT,g=4_t=6,power,-0.662865,-3.940222,0.34,...,1.192689,1.556540,0.759644,3.940222,1.0,0.0,0.914571,3.942386,0.464467,4.237540
8,canonical,B1_selection_obs,1,200,GATT,g=4_t=7,power,-0.638164,-3.935642,0.36,...,1.260237,1.674856,0.780911,3.935642,1.0,0.0,0.943542,3.938390,0.454061,3.879902


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_selection_obs,1,200,corrected,100,404.32,1.200035,0.338766,1.024670,...,0.257957,0.066399,0.274864,0.116879,0.33158,0.130188,0.991882,0.347871,1.213404,0.455574
1,canonical,B1_selection_obs,1,200,plain,100,404.32,4.025527,0.110942,3.933742,...,1.000395,0.016557,0.000000,0.000000,0.00000,0.000000,1.172677,0.058078,1.477138,0.071593
